In [ ]:
#!/usr/bin/env python3
"""
Standalone test script for ResNet34 3-Layer U-Net Land Cover Model
Load a trained checkpoint and evaluate on test data
"""
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
import time
import json

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# ============================================================
# CONFIGURATION
# ============================================================
CHECKPOINT_PATH = "checkpoints/resnet34-3layer-dropout0.4-wd0.0001-lr5e-4-epoch=32-val_miou=0.5375.ckpt"  
TEST_IMG_DIR = "data/test/images"  
TEST_MASK_DIR = "data/test/masks"
BATCH_SIZE = 16
SAVE_PREDICTIONS = True
OUTPUT_DIR = "test_predictions_3layer"
NUM_SAMPLES_TO_SHOW = 50  # Number of samples to visualise

# ============================================================
# DATASET
# ============================================================
class LandCoverDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        
        # Verify directories exist
        if not os.path.exists(img_dir):
            raise FileNotFoundError(f"Image directory not found: {img_dir}")
        if not os.path.exists(mask_dir):
            raise FileNotFoundError(f"Mask directory not found: {mask_dir}")
        
        # Get all image files
        all_images = sorted(os.listdir(img_dir))
        
        # Filter to only valid image files and check if masks exist
        self.images = []
        missing_masks = []
        
        for img_name in all_images:
            # Skip non-image files
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
                
            img_path = os.path.join(img_dir, img_name)
            
            # Determine mask filename
            if img_name.endswith('.jpg'):
                mask_name = img_name.replace(".jpg", "_m.png")
            elif img_name.endswith('.jpeg'):
                mask_name = img_name.replace(".jpeg", "_m.png")
            else:
                mask_name = img_name.replace(".png", "_m.png")
            
            mask_path = os.path.join(mask_dir, mask_name)
            
            # Check if both files exist
            if os.path.exists(img_path) and os.path.exists(mask_path):
                self.images.append(img_name)
            else:
                if not os.path.exists(mask_path):
                    missing_masks.append(mask_name)
        
        if len(self.images) == 0:
            raise ValueError(f"No valid image-mask pairs found in {img_dir}")
        
        print(f" Found {len(self.images)} valid test images")
        if missing_masks:
            print(f"  Warning: {len(missing_masks)} images skipped due to missing masks")
            if len(missing_masks) <= 5:
                print(f"   Missing masks: {missing_masks}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Determine mask filename
        if img_name.endswith('.jpg'):
            mask_name = img_name.replace(".jpg", "_m.png")
        elif img_name.endswith('.jpeg'):
            mask_name = img_name.replace(".jpeg", "_m.png")
        else:
            mask_name = img_name.replace(".png", "_m.png")
            
        mask_path = os.path.join(self.mask_dir, mask_name)

        # Read image
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Failed to read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Read mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Failed to read mask: {mask_path}")
        mask = np.clip(mask, 0, 4)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = torch.tensor(augmented['mask'], dtype=torch.long)
        else:
            image = image.astype(np.float32) / 255.0
            image = torch.tensor(image).permute(2, 0, 1)
            mask = torch.tensor(mask, dtype=torch.long)

        return image, mask

# ============================================================
# TRANSFORMS
# ============================================================
def get_test_transform():
    """Test transform - only normalization, no augmentation"""
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

# ============================================================
# MODEL ARCHITECTURE
# ============================================================
class ResNet34UNet3Layer(nn.Module):
    """ResNet34 U-Net with 3 encoder layers"""
    def __init__(self, n_classes=5, pretrained=True, dropout_p=0.4):
        super().__init__()

        try:
            resnet = models.resnet34(pretrained=pretrained)
        except TypeError:
            resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)

        # Encoder - Only use first 3 layers
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.encoder2 = resnet.layer1
        self.encoder3 = resnet.layer2

        # Decoder - 2 upsampling stages
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self._make_decoder_block(128, 64, dropout_p)

        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = self._make_decoder_block(128, 64, dropout_p)

        self.final_upsample = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.final = nn.Conv2d(64, n_classes, kernel_size=1)

    def _make_decoder_block(self, in_ch, out_ch, dropout_p):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p)
        )

    def forward(self, x):
        e1 = self.encoder1(x)
        e1_pooled = self.pool(e1)
        e2 = self.encoder2(e1_pooled)
        e3 = self.encoder3(e2)

        d2 = self.up2(e3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        d1 = self.final_upsample(d1)
        return self.final(d1)

# ============================================================
# LIGHTNING MODULE
# ============================================================
class LitUNet(pl.LightningModule):
    def __init__(self, n_classes=5, lr=5e-4, class_weights=None, dropout_p=0.4, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters(ignore=['class_weights'])
        
        self.model = ResNet34UNet3Layer(n_classes, pretrained=False, dropout_p=dropout_p)
        
        if class_weights is not None:
            self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        else:
            self.criterion = nn.CrossEntropyLoss()
        
        self.n_classes = n_classes

    def forward(self, x):
        return self.model(x)

# ============================================================
# VISUALIZATION
# ============================================================
class_info = {
    0: ("Background", (0, 0, 0)),
    1: ("Building",   (255, 0, 0)),
    2: ("Woodland",   (0, 255, 0)),
    3: ("Water",      (0, 0, 255)),
    4: ("Road",       (255, 255, 0)),
}

def id_to_color(mask):
    color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_id, (label, color) in class_info.items():
        color_mask[mask == class_id] = color
    return color_mask

def visualize_predictions(model, dataset, num_samples=50, save_dir=None):
    """Visualize model predictions"""
    print(f"\n Generating visualizations for {num_samples} samples...")
    
    model.eval()
    device = next(model.parameters()).device
    
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    for i in range(min(num_samples, len(dataset))):
        img, mask = dataset[i]
        img_name = dataset.images[i]

        with torch.no_grad():
            pred = model(img.unsqueeze(0).to(device))
        pred = torch.argmax(pred, dim=1).squeeze().cpu().numpy()
        mask_np = mask.cpu().numpy()

        # Denormalize image
        img_np = img.cpu().numpy().transpose(1, 2, 0)
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
        
        overlay_pred = id_to_color(pred)
        overlay_mask = id_to_color(mask_np)
        overlay_gt = cv2.addWeighted(img_np, 0.6, overlay_mask, 0.4, 0)
        overlay_pred_blend = cv2.addWeighted(img_np, 0.6, overlay_pred, 0.4, 0)

        unique, counts = np.unique(pred, return_counts=True)
        dominant_class = unique[np.argmax(counts)]
        dominant_label = class_info.get(dominant_class, ("Background", None))[0]

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f"{img_name}", fontsize=9)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(overlay_gt)
        axes[i, 1].set_title("Ground Truth", fontsize=9)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(overlay_pred_blend)
        axes[i, 2].set_title(f"Prediction ({dominant_label})", fontsize=9)
        axes[i, 2].axis('off')

        # Save individual predictions
        if save_dir is not None:
            save_path = os.path.join(save_dir, f"{os.path.splitext(img_name)[0]}_prediction_3layer_test.png")
            row_fig, row_axes = plt.subplots(1, 3, figsize=(12, 4))
            row_axes[0].imshow(img_np)
            row_axes[0].set_title(f"{img_name}", fontsize=9)
            row_axes[0].axis('off')
            row_axes[1].imshow(overlay_gt)
            row_axes[1].set_title("Ground Truth", fontsize=9)
            row_axes[1].axis('off')
            row_axes[2].imshow(overlay_pred_blend)
            row_axes[2].set_title(f"Prediction ({dominant_label})", fontsize=9)
            row_axes[2].axis('off')
            plt.tight_layout()
            plt.savefig(save_path, bbox_inches="tight")
            plt.close(row_fig)
        
        if (i + 1) % 10 == 0:
            print(f"   Processed {i + 1}/{num_samples} samples")

    legend_elements = [Patch(facecolor=np.array(color)/255.0, edgecolor='black', label=label)
                       for label, color in [v for v in class_info.values()]]
    fig.legend(handles=legend_elements, loc='upper right', title="Classes")
    plt.tight_layout()

    if save_dir is not None:
        combined_path = os.path.join(save_dir, "combined_predictions_3layer_test.png")
        plt.savefig(combined_path, bbox_inches="tight", dpi=150)
        print(f" Saved combined predictions to: {combined_path}")

    plt.show()

# ============================================================
# TEST FUNCTION
# ============================================================
def test_model(model, test_dataset, batch_size=16):
    """Test model and compute metrics"""
    print("\n" + "="*70)
    print("TESTING MODEL")
    print("="*70)
    
    model.eval()
    device = next(model.parameters()).device
    
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    all_preds = []
    all_masks = []
    test_loss = 0.0
    n_classes = model.n_classes
    
    print(f"Testing on {len(test_dataset)} samples...")
    
    with torch.no_grad():
        for batch_idx, (imgs, masks) in enumerate(test_loader):
            imgs = imgs.to(device)
            masks = masks.to(device)
            
            outputs = model(imgs)
            loss = model.criterion(outputs, masks)
            test_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.append(preds.cpu().numpy())
            all_masks.append(masks.cpu().numpy())
            
            if (batch_idx + 1) % 10 == 0:
                print(f"   Processed {batch_idx + 1}/{len(test_loader)} batches")
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)
    
    # Calculate metrics
    print("\n" + "="*70)
    print("COMPUTING METRICS")
    print("="*70)
    
    accuracy = (all_preds == all_masks).mean()
    
    intersection = np.zeros(n_classes)
    union = np.zeros(n_classes)
    dice_scores = np.zeros(n_classes)
    
    for c in range(n_classes):
        pred_c = all_preds == c
        mask_c = all_masks == c
        
        inter = np.logical_and(pred_c, mask_c).sum()
        union_c = np.logical_or(pred_c, mask_c).sum()
        
        intersection[c] = inter
        union[c] = union_c
        dice_scores[c] = (2 * inter) / (pred_c.sum() + mask_c.sum() + 1e-6)
    
    IoU = intersection / np.maximum(union, 1)
    mean_IoU = np.nanmean(IoU)
    mean_dice = np.nanmean(dice_scores)
    avg_test_loss = test_loss / len(test_loader)
    
    # Print results
    print(f"\n TEST RESULTS:")
    print(f"   Test Loss:     {avg_test_loss:.4f}")
    print(f"   Accuracy:      {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   Mean IoU:      {mean_IoU:.4f}")
    print(f"   Mean Dice:     {mean_dice:.4f}")
    
    print(f"\n PER-CLASS METRICS:")
    class_names = ["Background", "Building", "Woodland", "Water", "Road"]
    print(f"{'Class':<12} {'IoU':>8} {'Dice':>8}")
    print("-" * 30)
    for c in range(n_classes):
        print(f"{class_names[c]:<12} {IoU[c]:>8.4f} {dice_scores[c]:>8.4f}")
    
    print("\n" + "="*70)
    
    return {
        'test_loss': float(avg_test_loss),
        'accuracy': float(accuracy),
        'mean_iou': float(mean_IoU),
        'mean_dice': float(mean_dice),
        'per_class_iou': IoU.tolist(),
        'per_class_dice': dice_scores.tolist(),
        'class_names': class_names
    }

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    start_time = time.time()
    
    print("\n" + "="*70)
    print("RESNET34 3-LAYER U-NET - TEST SCRIPT")
    print("="*70)
    
    # Check if checkpoint exists
    if not os.path.exists(CHECKPOINT_PATH):
        print(f"\n❌ ERROR: Checkpoint not found at: {CHECKPOINT_PATH}")
        print("\nAvailable checkpoints:")
        checkpoint_dir = "checkpoints"
        if os.path.exists(checkpoint_dir):
            checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith('.ckpt')]
            if checkpoints:
                for i, ckpt in enumerate(checkpoints, 1):
                    print(f"   {i}. {ckpt}")
            else:
                print("   No checkpoints found!")
        else:
            print(f"   Checkpoint directory '{checkpoint_dir}' does not exist!")
        print("\nPlease update CHECKPOINT_PATH in the script.")
        exit(1)
    
    # Load model
    # Replace the checkpoint loading section in your main code with this:

    print(f"\n Loading checkpoint from: {CHECKPOINT_PATH}")
    try:
        # Load checkpoint
        checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
        
        # Remove problematic keys
        if 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
            keys_to_remove = [k for k in state_dict.keys() if k.startswith('criterion.')]
            for key in keys_to_remove:
                del state_dict[key]
                print(f"   Removed key: {key}")
        
        # Create model and load state dict
        model = LitUNet(n_classes=5)
        model.load_state_dict(checkpoint['state_dict'], strict=False)
        
        print(" Model loaded successfully!")
        
        # Move to GPU if available
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        print(f" Using device: {device}")
        
        # Count parameters
        total_params = sum(p.numel() for p in model.parameters())
        print(f" Model parameters: {total_params:,} ({total_params/1e6:.2f}M)")
        
    except Exception as e:
        print(f" ERROR loading checkpoint: {e}")
        import traceback
        traceback.print_exc()
        exit(1)
    
    # Load test dataset
    print(f"\n Loading test data from: {TEST_IMG_DIR}")
    if not os.path.exists(TEST_IMG_DIR):
        print(f" ERROR: Test image directory not found: {TEST_IMG_DIR}")
        exit(1)
    if not os.path.exists(TEST_MASK_DIR):
        print(f" ERROR: Test mask directory not found: {TEST_MASK_DIR}")
        exit(1)
    
    test_transform = get_test_transform()
    test_dataset = LandCoverDataset(TEST_IMG_DIR, TEST_MASK_DIR, transform=test_transform)
    
    # Run testing
    test_results = test_model(model, test_dataset, batch_size=BATCH_SIZE)
    
    # Save results to JSON
    results_file = "test_results_3layer.json"
    with open(results_file, 'w') as f:
        json.dump(test_results, f, indent=4)
    print(f"\n Test results saved to: {results_file}")
    
    # Generate visualizations
    if SAVE_PREDICTIONS:
        print(f"\n Generating prediction visualizations...")
        visualize_predictions(
            model, 
            test_dataset, 
            num_samples=min(NUM_SAMPLES_TO_SHOW, len(test_dataset)),
            save_dir=OUTPUT_DIR
        )
    
    # Print summary
    elapsed = time.time() - start_time
    print("\n" + "="*70)
    print("TESTING COMPLETE")
    print("="*70)
    print(f" Total time: {elapsed:.2f}s ({elapsed/60:.2f} minutes)")
    print(f" Results saved to: {results_file}")
    if SAVE_PREDICTIONS:
        print(f"Predictions saved to: {OUTPUT_DIR}/")
    print("="*70 + "\n")

In [ ]:
#!/usr/bin/env python3
"""
Standalone test script for ResNet34 4-Layer U-Net Land Cover Model
Load a trained checkpoint and evaluate on test data
"""
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
import time
import json

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# ============================================================
# CONFIGURATION
# ============================================================
CHECKPOINT_PATH = "checkpoints/resnet34-4layer-dropout0.2-wd0.0001-lr1e-3-epoch=40-val_miou=0.6120.ckpt"  
TEST_IMG_DIR = "data/test/images"
TEST_MASK_DIR = "data/test/masks"
BATCH_SIZE = 16
SAVE_PREDICTIONS = True
OUTPUT_DIR = "test_predictions_4layer"
NUM_SAMPLES_TO_SHOW = 50  # Number of samples to visualize

# ============================================================
# DATASET
# ============================================================
class LandCoverDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        
        # Verify directories exist
        if not os.path.exists(img_dir):
            raise FileNotFoundError(f"Image directory not found: {img_dir}")
        if not os.path.exists(mask_dir):
            raise FileNotFoundError(f"Mask directory not found: {mask_dir}")
        
        # Get all image files
        all_images = sorted(os.listdir(img_dir))
        
        # Filter to only valid image files and check if masks exist
        self.images = []
        missing_masks = []
        
        for img_name in all_images:
            # Skip non-image files
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
                
            img_path = os.path.join(img_dir, img_name)
            
            # Determine mask filename
            if img_name.endswith('.jpg'):
                mask_name = img_name.replace(".jpg", "_m.png")
            elif img_name.endswith('.jpeg'):
                mask_name = img_name.replace(".jpeg", "_m.png")
            else:
                mask_name = img_name.replace(".png", "_m.png")
            
            mask_path = os.path.join(mask_dir, mask_name)
            
            # Check if both files exist
            if os.path.exists(img_path) and os.path.exists(mask_path):
                self.images.append(img_name)
            else:
                if not os.path.exists(mask_path):
                    missing_masks.append(mask_name)
        
        if len(self.images) == 0:
            raise ValueError(f"No valid image-mask pairs found in {img_dir}")
        
        print(f" Found {len(self.images)} valid test images")
        if missing_masks:
            print(f"  Warning: {len(missing_masks)} images skipped due to missing masks")
            if len(missing_masks) <= 5:
                print(f"   Missing masks: {missing_masks}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Determine mask filename
        if img_name.endswith('.jpg'):
            mask_name = img_name.replace(".jpg", "_m.png")
        elif img_name.endswith('.jpeg'):
            mask_name = img_name.replace(".jpeg", "_m.png")
        else:
            mask_name = img_name.replace(".png", "_m.png")
            
        mask_path = os.path.join(self.mask_dir, mask_name)

        # Read image
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Failed to read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Read mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Failed to read mask: {mask_path}")
        mask = np.clip(mask, 0, 4)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = torch.tensor(augmented['mask'], dtype=torch.long)
        else:
            image = image.astype(np.float32) / 255.0
            image = torch.tensor(image).permute(2, 0, 1)
            mask = torch.tensor(mask, dtype=torch.long)

        return image, mask

# ============================================================
# TRANSFORMS
# ============================================================
def get_test_transform():
    """Test transform - only normalization, no augmentation"""
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

# ============================================================
# MODEL ARCHITECTURE
# ============================================================
class ResNet34UNet4Layer(nn.Module):
    """ResNet34 U-Net with 4 encoder layers"""
    def __init__(self, n_classes=5, pretrained=True, dropout_p=0.4):
        super().__init__()

        try:
            resnet = models.resnet34(pretrained=pretrained)
        except TypeError:
            resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)

        # Encoder - Use first 4 layers
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.encoder2 = resnet.layer1
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3

        # Decoder - 3 upsampling stages
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = self._make_decoder_block(256, 128, dropout_p)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self._make_decoder_block(128, 64, dropout_p)

        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = self._make_decoder_block(128, 64, dropout_p)

        self.final_upsample = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.final = nn.Conv2d(64, n_classes, kernel_size=1)

    def _make_decoder_block(self, in_ch, out_ch, dropout_p):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p)
        )

    def forward(self, x):
        e1 = self.encoder1(x)
        e1_pooled = self.pool(e1)
        e2 = self.encoder2(e1_pooled)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        d3 = self.up3(e4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        d1 = self.final_upsample(d1)
        return self.final(d1)

# ============================================================
# LIGHTNING MODULE
# ============================================================
class LitUNet(pl.LightningModule):
    def __init__(self, n_classes=5, lr=5e-4, class_weights=None, dropout_p=0.4, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters(ignore=['class_weights'])
        
        self.model = ResNet34UNet4Layer(n_classes, pretrained=False, dropout_p=dropout_p)
        
        if class_weights is not None:
            self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        else:
            self.criterion = nn.CrossEntropyLoss()
        
        self.n_classes = n_classes

    def forward(self, x):
        return self.model(x)

# ============================================================
# VISUALIZATION
# ============================================================
class_info = {
    0: ("Background", (0, 0, 0)),
    1: ("Building",   (255, 0, 0)),
    2: ("Woodland",   (0, 255, 0)),
    3: ("Water",      (0, 0, 255)),
    4: ("Road",       (255, 255, 0)),
}

def id_to_color(mask):
    color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_id, (label, color) in class_info.items():
        color_mask[mask == class_id] = color
    return color_mask

def visualize_predictions(model, dataset, num_samples=50, save_dir=None):
    """Visualize model predictions"""
    print(f"\n Generating visualizations for {num_samples} samples...")
    
    model.eval()
    device = next(model.parameters()).device
    
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    for i in range(min(num_samples, len(dataset))):
        img, mask = dataset[i]
        img_name = dataset.images[i]

        with torch.no_grad():
            pred = model(img.unsqueeze(0).to(device))
        pred = torch.argmax(pred, dim=1).squeeze().cpu().numpy()
        mask_np = mask.cpu().numpy()

        # Denormalize image
        img_np = img.cpu().numpy().transpose(1, 2, 0)
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
        
        overlay_pred = id_to_color(pred)
        overlay_mask = id_to_color(mask_np)
        overlay_gt = cv2.addWeighted(img_np, 0.6, overlay_mask, 0.4, 0)
        overlay_pred_blend = cv2.addWeighted(img_np, 0.6, overlay_pred, 0.4, 0)

        unique, counts = np.unique(pred, return_counts=True)
        dominant_class = unique[np.argmax(counts)]
        dominant_label = class_info.get(dominant_class, ("Background", None))[0]

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f"{img_name}", fontsize=9)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(overlay_gt)
        axes[i, 1].set_title("Ground Truth", fontsize=9)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(overlay_pred_blend)
        axes[i, 2].set_title(f"Prediction ({dominant_label})", fontsize=9)
        axes[i, 2].axis('off')

        # Save individual predictions
        if save_dir is not None:
            save_path = os.path.join(save_dir, f"{os.path.splitext(img_name)[0]}_prediction_4layer_test.png")
            row_fig, row_axes = plt.subplots(1, 3, figsize=(12, 4))
            row_axes[0].imshow(img_np)
            row_axes[0].set_title(f"{img_name}", fontsize=9)
            row_axes[0].axis('off')
            row_axes[1].imshow(overlay_gt)
            row_axes[1].set_title("Ground Truth", fontsize=9)
            row_axes[1].axis('off')
            row_axes[2].imshow(overlay_pred_blend)
            row_axes[2].set_title(f"Prediction ({dominant_label})", fontsize=9)
            row_axes[2].axis('off')
            plt.tight_layout()
            plt.savefig(save_path, bbox_inches="tight")
            plt.close(row_fig)
        
        if (i + 1) % 10 == 0:
            print(f"   Processed {i + 1}/{num_samples} samples")

    legend_elements = [Patch(facecolor=np.array(color)/255.0, edgecolor='black', label=label)
                       for label, color in [v for v in class_info.values()]]
    fig.legend(handles=legend_elements, loc='upper right', title="Classes")
    plt.tight_layout()

    if save_dir is not None:
        combined_path = os.path.join(save_dir, "combined_predictions_4layer_test.png")
        plt.savefig(combined_path, bbox_inches="tight", dpi=150)
        print(f" Saved combined predictions to: {combined_path}")

    plt.show()

# ============================================================
# TEST FUNCTION
# ============================================================
def test_model(model, test_dataset, batch_size=16):
    """Test model and compute metrics"""
    print("\n" + "="*70)
    print("TESTING MODEL")
    print("="*70)
    
    model.eval()
    device = next(model.parameters()).device
    
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    all_preds = []
    all_masks = []
    test_loss = 0.0
    n_classes = model.n_classes
    
    print(f"Testing on {len(test_dataset)} samples...")
    
    with torch.no_grad():
        for batch_idx, (imgs, masks) in enumerate(test_loader):
            imgs = imgs.to(device)
            masks = masks.to(device)
            
            outputs = model(imgs)
            loss = model.criterion(outputs, masks)
            test_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.append(preds.cpu().numpy())
            all_masks.append(masks.cpu().numpy())
            
            if (batch_idx + 1) % 10 == 0:
                print(f"   Processed {batch_idx + 1}/{len(test_loader)} batches")
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)
    
    # Calculate metrics
    print("\n" + "="*70)
    print("COMPUTING METRICS")
    print("="*70)
    
    accuracy = (all_preds == all_masks).mean()
    
    intersection = np.zeros(n_classes)
    union = np.zeros(n_classes)
    dice_scores = np.zeros(n_classes)
    
    for c in range(n_classes):
        pred_c = all_preds == c
        mask_c = all_masks == c
        
        inter = np.logical_and(pred_c, mask_c).sum()
        union_c = np.logical_or(pred_c, mask_c).sum()
        
        intersection[c] = inter
        union[c] = union_c
        dice_scores[c] = (2 * inter) / (pred_c.sum() + mask_c.sum() + 1e-6)
    
    IoU = intersection / np.maximum(union, 1)
    mean_IoU = np.nanmean(IoU)
    mean_dice = np.nanmean(dice_scores)
    avg_test_loss = test_loss / len(test_loader)
    
    # Print results
    print(f"\n TEST RESULTS:")
    print(f"   Test Loss:     {avg_test_loss:.4f}")
    print(f"   Accuracy:      {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   Mean IoU:      {mean_IoU:.4f}")
    print(f"   Mean Dice:     {mean_dice:.4f}")
    
    print(f"\n PER-CLASS METRICS:")
    class_names = ["Background", "Building", "Woodland", "Water", "Road"]
    print(f"{'Class':<12} {'IoU':>8} {'Dice':>8}")
    print("-" * 30)
    for c in range(n_classes):
        print(f"{class_names[c]:<12} {IoU[c]:>8.4f} {dice_scores[c]:>8.4f}")
    
    print("\n" + "="*70)
    
    return {
        'test_loss': float(avg_test_loss),
        'accuracy': float(accuracy),
        'mean_iou': float(mean_IoU),
        'mean_dice': float(mean_dice),
        'per_class_iou': IoU.tolist(),
        'per_class_dice': dice_scores.tolist(),
        'class_names': class_names
    }

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    start_time = time.time()
    
    print("\n" + "="*70)
    print("RESNET34 4-LAYER U-NET - TEST SCRIPT")
    print("="*70)
    
    # Check if checkpoint exists
    if not os.path.exists(CHECKPOINT_PATH):
        print(f"\n ERROR: Checkpoint not found at: {CHECKPOINT_PATH}")
        print("\nAvailable checkpoints:")
        checkpoint_dir = "checkpoints"
        if os.path.exists(checkpoint_dir):
            checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith('.ckpt')]
            if checkpoints:
                for i, ckpt in enumerate(checkpoints, 1):
                    print(f"   {i}. {ckpt}")
            else:
                print("   No checkpoints found!")
        else:
            print(f"   Checkpoint directory '{checkpoint_dir}' does not exist!")
        print("\nPlease update CHECKPOINT_PATH in the script.")
        exit(1)
    
    # Load model
    print(f"\n Loading checkpoint from: {CHECKPOINT_PATH}")
    try:
        # Load checkpoint
        checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
        
        # Remove problematic keys
        if 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
            keys_to_remove = [k for k in state_dict.keys() if k.startswith('criterion.')]
            for key in keys_to_remove:
                del state_dict[key]
                print(f"   Removed key: {key}")
        
        # Create model and load state dict
        model = LitUNet(n_classes=5)
        model.load_state_dict(checkpoint['state_dict'], strict=False)
        
        print(" Model loaded successfully!")
        
        # Move to GPU if available
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        print(f" Using device: {device}")
        
        # Count parameters
        total_params = sum(p.numel() for p in model.parameters())
        print(f" Model parameters: {total_params:,} ({total_params/1e6:.2f}M)")
        
    except Exception as e:
        print(f" ERROR loading checkpoint: {e}")
        import traceback
        traceback.print_exc()
        exit(1)
    
    # Load test dataset
    print(f"\n Loading test data from: {TEST_IMG_DIR}")
    if not os.path.exists(TEST_IMG_DIR):
        print(f" ERROR: Test image directory not found: {TEST_IMG_DIR}")
        exit(1)
    if not os.path.exists(TEST_MASK_DIR):
        print(f" ERROR: Test mask directory not found: {TEST_MASK_DIR}")
        exit(1)
    
    test_transform = get_test_transform()
    test_dataset = LandCoverDataset(TEST_IMG_DIR, TEST_MASK_DIR, transform=test_transform)
    
    # Run testing
    test_results = test_model(model, test_dataset, batch_size=BATCH_SIZE)
    
    # Save results to JSON
    results_file = "test_results_4layer.json"
    with open(results_file, 'w') as f:
        json.dump(test_results, f, indent=4)
    print(f"\n Test results saved to: {results_file}")
    
    # Generate visualizations
    if SAVE_PREDICTIONS:
        print(f"\n Generating prediction visualizations...")
        visualize_predictions(
            model, 
            test_dataset, 
            num_samples=min(NUM_SAMPLES_TO_SHOW, len(test_dataset)),
            save_dir=OUTPUT_DIR
        )
    
    # Print summary
    elapsed = time.time() - start_time
    print("\n" + "="*70)
    print("TESTING COMPLETE")
    print("="*70)
    print(f" Total time: {elapsed:.2f}s ({elapsed/60:.2f} minutes)")
    print(f" Results saved to: {results_file}")
    if SAVE_PREDICTIONS:
        print(f" Predictions saved to: {OUTPUT_DIR}/")
    print("="*70 + "\n")

In [ ]:
#!/usr/bin/env python3
"""
Standalone test script for ResNet34 5-Layer U-Net Land Cover Model
Load a trained checkpoint and evaluate on test data
"""
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
import time
import json

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# ============================================================
# CONFIGURATION
# ============================================================
CHECKPOINT_PATH = "checkpoints/resnet34-5layer-dropout0.2-wd0.0001-lr1e-3-epoch=49-val_miou=0.6038.ckpt"  
TEST_IMG_DIR = "data/test/images"
TEST_MASK_DIR = "data/test/masks"
BATCH_SIZE = 16
SAVE_PREDICTIONS = True
OUTPUT_DIR = "test_predictions_5layer"
NUM_SAMPLES_TO_SHOW = 50  # Number of samples to visualize

# ============================================================
# DATASET
# ============================================================
class LandCoverDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        
        # Verify directories exist
        if not os.path.exists(img_dir):
            raise FileNotFoundError(f"Image directory not found: {img_dir}")
        if not os.path.exists(mask_dir):
            raise FileNotFoundError(f"Mask directory not found: {mask_dir}")
        
        # Get all image files
        all_images = sorted(os.listdir(img_dir))
        
        # Filter to only valid image files and check if masks exist
        self.images = []
        missing_masks = []
        
        for img_name in all_images:
            # Skip non-image files
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
                
            img_path = os.path.join(img_dir, img_name)
            
            # Determine mask filename
            if img_name.endswith('.jpg'):
                mask_name = img_name.replace(".jpg", "_m.png")
            elif img_name.endswith('.jpeg'):
                mask_name = img_name.replace(".jpeg", "_m.png")
            else:
                mask_name = img_name.replace(".png", "_m.png")
            
            mask_path = os.path.join(mask_dir, mask_name)
            
            # Check if both files exist
            if os.path.exists(img_path) and os.path.exists(mask_path):
                self.images.append(img_name)
            else:
                if not os.path.exists(mask_path):
                    missing_masks.append(mask_name)
        
        if len(self.images) == 0:
            raise ValueError(f"No valid image-mask pairs found in {img_dir}")
        
        print(f" Found {len(self.images)} valid test images")
        if missing_masks:
            print(f"  Warning: {len(missing_masks)} images skipped due to missing masks")
            if len(missing_masks) <= 5:
                print(f"   Missing masks: {missing_masks}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Determine mask filename
        if img_name.endswith('.jpg'):
            mask_name = img_name.replace(".jpg", "_m.png")
        elif img_name.endswith('.jpeg'):
            mask_name = img_name.replace(".jpeg", "_m.png")
        else:
            mask_name = img_name.replace(".png", "_m.png")
            
        mask_path = os.path.join(self.mask_dir, mask_name)

        # Read image
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Failed to read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Read mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Failed to read mask: {mask_path}")
        mask = np.clip(mask, 0, 4)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = torch.tensor(augmented['mask'], dtype=torch.long)
        else:
            image = image.astype(np.float32) / 255.0
            image = torch.tensor(image).permute(2, 0, 1)
            mask = torch.tensor(mask, dtype=torch.long)

        return image, mask

# ============================================================
# TRANSFORMS
# ============================================================
def get_test_transform():
    """Test transform - only normalization, no augmentation"""
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

# ============================================================
# MODEL ARCHITECTURE
# ============================================================
class ResNet34UNet5Layer(nn.Module):
    """ResNet34 U-Net with all 5 encoder layers (full ResNet34)"""
    def __init__(self, n_classes=5, pretrained=True, dropout_p=0.4):
        super().__init__()

        try:
            resnet = models.resnet34(pretrained=pretrained)
        except TypeError:
            resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)

        # Encoder - Use all 5 layers
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.encoder2 = resnet.layer1
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3
        self.encoder5 = resnet.layer4

        # Decoder - 4 upsampling stages
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = self._make_decoder_block(512, 256, dropout_p)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = self._make_decoder_block(256, 128, dropout_p)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self._make_decoder_block(128, 64, dropout_p)

        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = self._make_decoder_block(128, 64, dropout_p)

        self.final_upsample = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.final = nn.Conv2d(64, n_classes, kernel_size=1)

    def _make_decoder_block(self, in_ch, out_ch, dropout_p):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout_p)
        )

    def forward(self, x):
        e1 = self.encoder1(x)
        e1_pooled = self.pool(e1)
        e2 = self.encoder2(e1_pooled)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        e5 = self.encoder5(e4)

        d4 = self.up4(e5)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        d1 = self.final_upsample(d1)
        return self.final(d1)

# ============================================================
# LIGHTNING MODULE
# ============================================================
class LitUNet(pl.LightningModule):
    def __init__(self, n_classes=5, lr=5e-4, class_weights=None, dropout_p=0.4, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters(ignore=['class_weights'])
        
        self.model = ResNet34UNet5Layer(n_classes, pretrained=False, dropout_p=dropout_p)
        
        if class_weights is not None:
            self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        else:
            self.criterion = nn.CrossEntropyLoss()
        
        self.n_classes = n_classes

    def forward(self, x):
        return self.model(x)

# ============================================================
# VISUALIZATION
# ============================================================
class_info = {
    0: ("Background", (0, 0, 0)),
    1: ("Building",   (255, 0, 0)),
    2: ("Woodland",   (0, 255, 0)),
    3: ("Water",      (0, 0, 255)),
    4: ("Road",       (255, 255, 0)),
}

def id_to_color(mask):
    color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_id, (label, color) in class_info.items():
        color_mask[mask == class_id] = color
    return color_mask

def visualize_predictions(model, dataset, num_samples=50, save_dir=None):
    """Visualize model predictions"""
    print(f"\n Generating visualizations for {num_samples} samples...")
    
    model.eval()
    device = next(model.parameters()).device
    
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    for i in range(min(num_samples, len(dataset))):
        img, mask = dataset[i]
        img_name = dataset.images[i]

        with torch.no_grad():
            pred = model(img.unsqueeze(0).to(device))
        pred = torch.argmax(pred, dim=1).squeeze().cpu().numpy()
        mask_np = mask.cpu().numpy()

        # Denormalize image
        img_np = img.cpu().numpy().transpose(1, 2, 0)
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
        
        overlay_pred = id_to_color(pred)
        overlay_mask = id_to_color(mask_np)
        overlay_gt = cv2.addWeighted(img_np, 0.6, overlay_mask, 0.4, 0)
        overlay_pred_blend = cv2.addWeighted(img_np, 0.6, overlay_pred, 0.4, 0)

        unique, counts = np.unique(pred, return_counts=True)
        dominant_class = unique[np.argmax(counts)]
        dominant_label = class_info.get(dominant_class, ("Background", None))[0]

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f"{img_name}", fontsize=9)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(overlay_gt)
        axes[i, 1].set_title("Ground Truth", fontsize=9)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(overlay_pred_blend)
        axes[i, 2].set_title(f"Prediction ({dominant_label})", fontsize=9)
        axes[i, 2].axis('off')

        # Save individual predictions
        if save_dir is not None:
            save_path = os.path.join(save_dir, f"{os.path.splitext(img_name)[0]}_prediction_5layer_test.png")
            row_fig, row_axes = plt.subplots(1, 3, figsize=(12, 4))
            row_axes[0].imshow(img_np)
            row_axes[0].set_title(f"{img_name}", fontsize=9)
            row_axes[0].axis('off')
            row_axes[1].imshow(overlay_gt)
            row_axes[1].set_title("Ground Truth", fontsize=9)
            row_axes[1].axis('off')
            row_axes[2].imshow(overlay_pred_blend)
            row_axes[2].set_title(f"Prediction ({dominant_label})", fontsize=9)
            row_axes[2].axis('off')
            plt.tight_layout()
            plt.savefig(save_path, bbox_inches="tight")
            plt.close(row_fig)
        
        if (i + 1) % 10 == 0:
            print(f"   Processed {i + 1}/{num_samples} samples")

    legend_elements = [Patch(facecolor=np.array(color)/255.0, edgecolor='black', label=label)
                       for label, color in [v for v in class_info.values()]]
    fig.legend(handles=legend_elements, loc='upper right', title="Classes")
    plt.tight_layout()

    if save_dir is not None:
        combined_path = os.path.join(save_dir, "combined_predictions_5layer_test.png")
        plt.savefig(combined_path, bbox_inches="tight", dpi=150)
        print(f" Saved combined predictions to: {combined_path}")

    plt.show()

# ============================================================
# TEST FUNCTION
# ============================================================
def test_model(model, test_dataset, batch_size=16):
    """Test model and compute metrics"""
    print("\n" + "="*70)
    print("TESTING MODEL")
    print("="*70)
    
    model.eval()
    device = next(model.parameters()).device
    
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    all_preds = []
    all_masks = []
    test_loss = 0.0
    n_classes = model.n_classes
    
    print(f"Testing on {len(test_dataset)} samples...")
    
    with torch.no_grad():
        for batch_idx, (imgs, masks) in enumerate(test_loader):
            imgs = imgs.to(device)
            masks = masks.to(device)
            
            outputs = model(imgs)
            loss = model.criterion(outputs, masks)
            test_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.append(preds.cpu().numpy())
            all_masks.append(masks.cpu().numpy())
            
            if (batch_idx + 1) % 10 == 0:
                print(f"   Processed {batch_idx + 1}/{len(test_loader)} batches")
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)
    
    # Calculate metrics
    print("\n" + "="*70)
    print("COMPUTING METRICS")
    print("="*70)
    
    accuracy = (all_preds == all_masks).mean()
    
    intersection = np.zeros(n_classes)
    union = np.zeros(n_classes)
    dice_scores = np.zeros(n_classes)
    
    for c in range(n_classes):
        pred_c = all_preds == c
        mask_c = all_masks == c
        
        inter = np.logical_and(pred_c, mask_c).sum()
        union_c = np.logical_or(pred_c, mask_c).sum()
        
        intersection[c] = inter
        union[c] = union_c
        dice_scores[c] = (2 * inter) / (pred_c.sum() + mask_c.sum() + 1e-6)
    
    IoU = intersection / np.maximum(union, 1)
    mean_IoU = np.nanmean(IoU)
    mean_dice = np.nanmean(dice_scores)
    avg_test_loss = test_loss / len(test_loader)
    
    # Print results
    print(f"\n TEST RESULTS:")
    print(f"   Test Loss:     {avg_test_loss:.4f}")
    print(f"   Accuracy:      {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   Mean IoU:      {mean_IoU:.4f}")
    print(f"   Mean Dice:     {mean_dice:.4f}")
    
    print(f"\n PER-CLASS METRICS:")
    class_names = ["Background", "Building", "Woodland", "Water", "Road"]
    print(f"{'Class':<12} {'IoU':>8} {'Dice':>8}")
    print("-" * 30)
    for c in range(n_classes):
        print(f"{class_names[c]:<12} {IoU[c]:>8.4f} {dice_scores[c]:>8.4f}")
    
    print("\n" + "="*70)
    
    return {
        'test_loss': float(avg_test_loss),
        'accuracy': float(accuracy),
        'mean_iou': float(mean_IoU),
        'mean_dice': float(mean_dice),
        'per_class_iou': IoU.tolist(),
        'per_class_dice': dice_scores.tolist(),
        'class_names': class_names
    }

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    start_time = time.time()
    
    print("\n" + "="*70)
    print("RESNET34 5-LAYER U-NET - TEST SCRIPT")
    print("="*70)
    
    # Check if checkpoint exists
    if not os.path.exists(CHECKPOINT_PATH):
        print(f"\n ERROR: Checkpoint not found at: {CHECKPOINT_PATH}")
        print("\nAvailable checkpoints:")
        checkpoint_dir = "checkpoints"
        if os.path.exists(checkpoint_dir):
            checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith('.ckpt')]
            if checkpoints:
                for i, ckpt in enumerate(checkpoints, 1):
                    print(f"   {i}. {ckpt}")
            else:
                print("   No checkpoints found!")
        else:
            print(f"   Checkpoint directory '{checkpoint_dir}' does not exist!")
        print("\nPlease update CHECKPOINT_PATH in the script.")
        exit(1)
    
    # Load model
    print(f"\n Loading checkpoint from: {CHECKPOINT_PATH}")
    try:
        # Load checkpoint
        checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
        
        # Remove problematic keys
        if 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
            keys_to_remove = [k for k in state_dict.keys() if k.startswith('criterion.')]
            for key in keys_to_remove:
                del state_dict[key]
                print(f"   Removed key: {key}")
        
        # Create model and load state dict
        model = LitUNet(n_classes=5)
        model.load_state_dict(checkpoint['state_dict'], strict=False)
        
        print(" Model loaded successfully!")
        
        # Move to GPU if available
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        print(f" Using device: {device}")
        
        # Count parameters
        total_params = sum(p.numel() for p in model.parameters())
        print(f" Model parameters: {total_params:,} ({total_params/1e6:.2f}M)")
        
    except Exception as e:
        print(f" ERROR loading checkpoint: {e}")
        import traceback
        traceback.print_exc()
        exit(1)
    
    # Load test dataset
    print(f"\n Loading test data from: {TEST_IMG_DIR}")
    if not os.path.exists(TEST_IMG_DIR):
        print(f" ERROR: Test image directory not found: {TEST_IMG_DIR}")
        exit(1)
    if not os.path.exists(TEST_MASK_DIR):
        print(f" ERROR: Test mask directory not found: {TEST_MASK_DIR}")
        exit(1)
    
    test_transform = get_test_transform()
    test_dataset = LandCoverDataset(TEST_IMG_DIR, TEST_MASK_DIR, transform=test_transform)
    
    # Run testing
    test_results = test_model(model, test_dataset, batch_size=BATCH_SIZE)
    
    # Save results to JSON
    results_file = "test_results_5layer.json"
    with open(results_file, 'w') as f:
        json.dump(test_results, f, indent=4)
    print(f"\n Test results saved to: {results_file}")
    
    # Generate visualizations
    if SAVE_PREDICTIONS:
        print(f"\n Generating prediction visualizations...")
        visualize_predictions(
            model, 
            test_dataset, 
            num_samples=min(NUM_SAMPLES_TO_SHOW, len(test_dataset)),
            save_dir=OUTPUT_DIR
        )
    
    # Print summary
    elapsed = time.time() - start_time
    print("\n" + "="*70)
    print("TESTING COMPLETE")
    print("="*70)
    print(f" Total time: {elapsed:.2f}s ({elapsed/60:.2f} minutes)")
    print(f" Results saved to: {results_file}")
    if SAVE_PREDICTIONS:
        print(f"Predictions saved to: {OUTPUT_DIR}/")
    print("="*70 + "\n")